# 01 — Excel Reader: Old vs. New

Side-by-side comparison of the **broken** pattern (removed in DBR 18.x) vs. the **native** Databricks Excel reader (DBR 17.1+).

In [1]:
from databricks.connect import DatabricksSession
from dotenv import load_dotenv
import os

load_dotenv()

spark = DatabricksSession.builder.serverless().getOrCreate()

UC_CATALOG = os.getenv("UC_CATALOG", "alexander_booth")
UC_SCHEMA  = os.getenv("UC_SCHEMA",  "salt_river_excel")
UC_VOLUME  = os.getenv("UC_VOLUME",  "raw_files")

FILE_PATH  = f"/Volumes/{UC_CATALOG}/{UC_SCHEMA}/{UC_VOLUME}/salt_river_fields.xlsx"

## ❌ The Old Way (broken)

This is what customers were running. On earlier DBR 18.0 Serverless it raised `AnalysisException: Failed to find data source: excel` (Crealytics JAR removed). On current Serverless the native reader handles `format("excel")` — but the old Crealytics options (`header=True`, `inferSchema=False`) are unrecognized, causing **0 rows, 0 cols** returned silently.

In [2]:
# Old Crealytics-style options — broken on current Serverless even though format("excel") no longer raises.
sheet_name = "Attendance"
try:
    broken_df = (
        spark.read
        .format("excel")
        .option("dataAddress", f"{sheet_name}!A1:G1000")
        .option("header", False)
        .option("inferSchema", False)
        .option("treatEmptyValuesAsNulls", False)
        .load(FILE_PATH)
    )
    print(f"[BROKEN] Rows: {broken_df.count()}  Cols: {len(broken_df.columns)}")
    broken_df.show(5)
except Exception as e:
    print(f"[EXPECTED ERROR] {type(e).__name__}: {e}")

[BROKEN] Rows: 1000  Cols: 7
+----------+--------------------+--------+----------+--------+-----------------+----------------+
|       _c0|                 _c1|     _c2|       _c3|     _c4|              _c5|             _c6|
+----------+--------------------+--------+----------+--------+-----------------+----------------+
| game_date|           home_team|opponent|attendance|capacity|         pct_full|gate_revenue_usd|
|2025-02-22|    Colorado Rockies|  Padres|      9819|   11000|             89.3|        187673.6|
|2025-02-24|Arizona Diamondbacks|    Cubs|     10237|   11000|93.09999999999999|       212420.75|
|2025-02-26|    Colorado Rockies| Brewers|      8114|   11000|             73.8|       157374.11|
|2025-02-28|Arizona Diamondbacks|  Padres|      7619|   11000|             69.3|       188699.73|
+----------+--------------------+--------+----------+--------+-----------------+----------------+
only showing top 5 rows


## ✅ The New Way — Native Excel Reader (DBR 17.1+)

`spark.read.excel()` is first-party. No JARs, no Maven packages, no pandas conversion.

In [3]:
attendance_df = (
    spark.read
    .option("dataAddress", sheet_name)
    .option("headerRows", 1)
    .excel(FILE_PATH)
)

print(f"Rows: {attendance_df.count()}  Cols: {len(attendance_df.columns)}")
attendance_df.printSchema()

Rows: 18  Cols: 7
root
 |-- game_date: string (nullable = true)
 |-- home_team: string (nullable = true)
 |-- opponent: string (nullable = true)
 |-- attendance: long (nullable = true)
 |-- capacity: long (nullable = true)
 |-- pct_full: decimal(16,14) (nullable = true)
 |-- gate_revenue_usd: decimal(8,2) (nullable = true)



In [4]:
attendance_df.show(10, truncate=False)

+----------+--------------------+---------+----------+--------+-----------------+----------------+
|game_date |home_team           |opponent |attendance|capacity|pct_full         |gate_revenue_usd|
+----------+--------------------+---------+----------+--------+-----------------+----------------+
|2025-02-22|Colorado Rockies    |Padres   |9819      |11000   |89.30000000000000|187673.60       |
|2025-02-24|Arizona Diamondbacks|Cubs     |10237     |11000   |93.09999999999999|212420.75       |
|2025-02-26|Colorado Rockies    |Brewers  |8114      |11000   |73.80000000000000|157374.11       |
|2025-02-28|Arizona Diamondbacks|Padres   |7619      |11000   |69.30000000000000|188699.73       |
|2025-03-02|Colorado Rockies    |White Sox|10854     |11000   |98.70000000000000|254566.08       |
|2025-03-04|Arizona Diamondbacks|Mariners |9618      |11000   |87.40000000000001|213704.44       |
|2025-03-06|Colorado Rockies    |Astros   |7322      |11000   |66.59999999999999|138656.37       |
|2025-03-0

## SQL equivalent via `read_files()`

In [5]:
spark.sql(f"""
    SELECT home_team, SUM(CAST(attendance AS LONG)) AS total_attendance
    FROM read_files('{FILE_PATH}', format => 'excel', dataAddress => 'Attendance', headerRows => 1)
    GROUP BY home_team
    ORDER BY total_attendance DESC
""").show()

+--------------------+----------------+
|           home_team|total_attendance|
+--------------------+----------------+
|    Colorado Rockies|           83750|
|Arizona Diamondbacks|           81022|
+--------------------+----------------+

